# Sysmon Malware Detection — เทรนบน Colab

โค้ดจริงอยู่ใน `host/ml_train.py` + `host/ml_predict.py` ของรีโป
notebook นี้แค่เรียกใช้ ไม่ได้ก๊อปโค้ดมาซ้ำ (จะได้ไม่หลุด sync กับรีโป)

**ต้องเตรียมเอง 1 อย่าง**: `merged_dataset.csv` (~50 MB) ไม่ได้อยู่ใน git
(`host/dataset/` ถูก gitignore) ต้องอัปโหลดเองในเซลล์ที่ 3

ไม่ต้อง `pip install` อะไรเลย — Colab มี pandas / scikit-learn / joblib ครบแล้ว

In [ ]:
# 1) เช็คเวอร์ชัน — โค้ดทดสอบแล้วบน pandas 2.x และ 3.x, sklearn 1.6-1.9
import sys, pandas, sklearn, numpy, joblib
print(sys.version)
for m in (pandas, sklearn, numpy, joblib):
    print(f"{m.__name__:12s} {m.__version__}")

In [ ]:
# 2) ดึงโค้ดจากรีโป
!git clone -q https://github.com/Anochaakimbo/sysmon_lab_environment.git repo || (cd repo && git pull -q)
%cd repo
!ls host/ml_*.py

In [ ]:
# 3) อัปโหลด merged_dataset.csv  (เลือกวิธีไหนก็ได้)
import os
os.makedirs("host/dataset", exist_ok=True)

# --- วิธี A: อัปโหลดจากเครื่อง (ช้าหน่อยเพราะ 50 MB) ---
from google.colab import files
up = files.upload()
for name in up:
    os.replace(name, "host/dataset/merged_dataset.csv")

# --- วิธี B: จาก Google Drive (เร็วกว่า ถ้าเอาไฟล์ขึ้น Drive ไว้แล้ว) ---
# from google.colab import drive; drive.mount('/content/drive')
# !cp "/content/drive/MyDrive/merged_dataset.csv" host/dataset/merged_dataset.csv

!ls -lh host/dataset/merged_dataset.csv

## เทรน

3 โปรโตคอลที่ควรรายงานในธีสิส (ตัวเลขต่างกันได้ 40 จุดบนข้อมูลชุดเดียวกัน
ถ้าไม่ระบุว่าใช้แบบไหน ตัวเลขไม่มีความหมาย):

| split | คืออะไร |
|-------|---------|
| `process` | GroupSplit ตาม ProcessGuid — **ตัวเลขหลัก** |
| `scenario` | leave-scenario-out — ไม่เคยเห็นการโจมตีชนิดนั้น (zero-day) |
| `paper` | ตาม Fig.4 ของเปเปอร์ — unsupervised เทรน benign ล้วน |

ผลต่อท้ายลง `reference/ml_results.csv` ทุกครั้ง โมเดลลง `models/*.pkl`

In [ ]:
# 4) ตัวเลขหลัก — process-level, group by ProcessGuid, + โมเดลบอกตระกูลมัลแวร์
!python host/ml_train.py --level process --split process --family

In [ ]:
# 5) เทียบกับเปเปอร์ตรงๆ — unsupervised เทรนด้วย benign อย่างเดียว
#    ดู AUC เป็นหลัก ไม่ใช่ F1 (test set เป็น malware 83% -> baseline flag ทุกแถวได้ F1 0.908)
!python host/ml_train.py --level event --split paper

In [ ]:
# 6) zero-day — เทรนด้วยการโจมตีชุดหนึ่ง แล้วเทสกับชนิดที่ไม่เคยเห็น
!python host/ml_train.py --level process --split scenario

In [ ]:
# 7) รวมผลทั้งหมดเป็นตารางเดียว (เอาไปวางในธีสิสได้)
import pandas as pd
r = pd.read_csv("reference/ml_results.csv")
cols = ["Learning_Type", "Protocol", "Level", "Algorithm",
        "Accuracy", "Precision", "Recall", "F1", "AUC"]
display(r[cols].round(4))

# เทียบกับเปเปอร์ (Achmad et al. 2025)
paper = pd.DataFrame([
    ("Supervised", "Naive Bayes", 0.3810), ("Supervised", "Decision Tree", 0.8720),
    ("Supervised", "Random Forest", 0.8868), ("Supervised", "SVM", 0.5906),
    ("Unsupervised", "Isolation Forest", 0.7620),
    ("Unsupervised", "Local Outlier Factor", 0.9750),
    ("Unsupervised", "One-Class SVM", 0.8051),
], columns=["Learning_Type", "Algorithm", "F1_paper"])
display(r.merge(paper, on=["Learning_Type", "Algorithm"])
         [["Protocol", "Level", "Algorithm", "F1", "F1_paper", "AUC"]].round(4))

## เอาโมเดลไปตรวจ log ที่ไม่ได้อยู่ในชุด train

In [ ]:
# 8) อัปโหลด CSV ที่จะตรวจ แล้วยิงเข้าโมเดล
from google.colab import files
up = files.upload()
target = list(up)[0]

!python host/ml_predict.py models/process_process.pkl "{target}" --out pred.csv

import pandas as pd
pred = pd.read_csv("pred.csv")
display(pred.sort_values("score", ascending=False).head(20))

In [ ]:
# 9) ดาวน์โหลดโมเดล + ผลกลับเครื่อง
!zip -qr out.zip models reference/ml_results.csv
from google.colab import files
files.download("out.zip")

## หมายเหตุ

- **`--split random` ห้ามใช้รายงาน** — event หลายแถวมาจาก process เดียวกัน
  สุ่มแล้วรั่วข้าม train/test ได้ F1 0.98 ปลอมๆ (เปเปอร์ใช้แบบนี้)
- **ดู `[baseline] flag ทุกแถว` ทุกครั้ง** — โมเดล unsupervised ที่ไม่ชนะบรรทัดนี้
  แปลว่าไม่ได้ตรวจจับอะไรเลย
- **SVM (rbf) ช้า** — ถูกจำกัดแถวเทรนไว้ที่ 20,000 ปรับด้วย `--svm-max-train`
- ถ้าอยากได้ตัวเลขที่ CPU ของ Colab ไหว ให้เริ่มจาก `--level process`
  (23,971 แถว) ก่อน `--level event` (71,086 แถว)
- รายละเอียดว่าทำไมโปรโตคอลถึงสำคัญ: `docs/ml_results_review.md`